# IRWA Final Project — Part 3: Ranking & Filtering

Every method uses the same AND filter from Part 2 (`conjunctive_search`): a product is returned only
if it contains all the query terms. Only the order of the results changes.

**Run from the repository root or from this folder.** Required files: `data/processed_corpus.pkl`
(output of Part 1) and `project_progress/part_2/test_queries_to_label.csv`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "myapp").is_dir())
sys.path.insert(0, str(ROOT))

from myapp.core.preprocessing import preprocess_text
from myapp.search.algorithms import create_index, conjunctive_search
from myapp.search.ranking import create_ranking_data, search_ranked

DATA = ROOT / "data"
PART2 = ROOT / "project_progress" / "part_2"
pd.set_option("display.max_colwidth", 60)

In [2]:
corpus = pd.read_pickle(DATA / "processed_corpus.pkl")
index, tf, idf = create_index(corpus)
data = create_ranking_data(corpus, idf)
title = dict(zip(corpus.pid, corpus.title))
print(f"{len(corpus):,} products, average length {data['avg_length']:.1f} tokens")

28,080 products, average length 39.1 tokens


In [3]:
test_labels = pd.read_csv(PART2 / "test_queries_to_label.csv")
TEST_QUERIES = test_labels.drop_duplicates("query_id").set_index("query_id")["query"].to_dict()
TEST_QUERIES

{1: 'men cotton polo tshirt',
 2: 'women cotton kurta',
 3: 'men hooded sweatshirt',
 4: 'men running shoes',
 5: 'women blue denim jacket'}

## 1. TF-IDF + cosine similarity and BM25

Both are in `myapp/search/ranking.py` and use the formulas of the lectures.

**TF-IDF + cosine**

* weight of a term in a text: $w_{t} = (1 + \log_2 f_t) \cdot idf_t$
* score: $\cos(q, d) = \dfrac{\sum_t w_{t,q} \cdot w_{t,d}}{\lVert q \rVert \cdot \lVert d \rVert}$

$\lVert d \rVert$ is the length of the **whole** tf-idf vector of the product, with all its terms.
In Part 2 we only divided tf by the length of the count vector, so that was not an exact cosine.

**BM25**

$$RSV_d = \sum_{t \in q} idf_t \cdot \frac{(k_1 + 1)\, tf_{t,d}}{k_1\left((1 - b) + b \cdot \frac{L_d}{L_{ave}}\right) + tf_{t,d}}$$

* $L_d$ = number of tokens of the product, $L_{ave}$ = average over the corpus
* $k_1 = 1.2$: repeating a term helps less and less (saturation)
* $b = 0.75$: how much long products are penalised

`idf` is the one from Part 2, $\log(N/df)$. Using another log base multiplies every score by the
same number, so the ranking does not change.

### Top 20 of our five queries

In [4]:
def top20(query, method):
    #results table [rank, pid, score, title, our label from Part 2 or NaN if not labelled]
    labels = test_labels[test_labels["query"] == query].set_index("pid")["label"]
    res = pd.DataFrame(search_ranked(query, index, idf, data, method=method, k=20), columns=["pid", "score"])
    res.insert(0, "rank", np.arange(1, len(res) + 1))
    res["title"] = res.pid.map(title)
    res["label"] = res.pid.map(labels)
    return res.round({"score": 4})

results = {(qid, m): top20(q, m) for qid, q in TEST_QUERIES.items() for m in ["tfidf", "bm25"]}

In [5]:
for qid, q in TEST_QUERIES.items():
    for method, name in [("tfidf", "TF-IDF + cosine"), ("bm25", "BM25")]:
        print(f"Query {qid}: {q} | {name}")
        display(results[(qid, method)])

Query 1: men cotton polo tshirt | TF-IDF + cosine


,rank,pid,score,title,label
0,1,TSHFUGZ9ZG5XN2FY,0.7847,Solid Men Polo Neck Blue T-Shirt,1.0
1,2,TSHFUGNWKWGPZ32S,0.7817,Solid Men Polo Neck Black T-Shirt,1.0
2,3,TSHFX3HD76TRJT3T,0.6706,Solid Men Polo Neck Grey T-Shirt,NaN
3,4,TSHFZZ2WHKH4EGKZ,0.6601,Solid Men Polo Neck Blue T-Shirt,NaN
4,5,TSHFUGKSZHBWZZ5E,0.6583,Solid Men Polo Neck Multicolor T-Shirt,1.0
5,6,TSHFDCH4EHXXBRRC,0.6527,Solid Men Polo Neck Black T-Shirt,NaN
6,7,TSHFZBYXXZVG4YG7,0.6527,Solid Men Polo Neck Black T-Shirt,NaN
7,8,TSHFUNN3HZKFZZDJ,0.6448,Printed Men Polo Neck Grey T-Shirt,NaN
8,9,TSHFP5Y5HYSMPNEV,0.6390,Printed Men Polo Neck Black T-Shirt,NaN
9,10,TSHEQFQ2FRKWAFZC,0.6365,Solid Men Polo Neck Black T-Shirt,NaN


Query 1: men cotton polo tshirt | BM25


,rank,pid,score,title,label
0,1,TSHFUNN3HZKFZZDJ,6.8448,Printed Men Polo Neck Grey T-Shirt,NaN
1,2,TSHFUNN36MB6BAPG,6.8012,Solid Men Polo Neck Yellow T-Shirt,NaN
2,3,TSHFUNN2YHM6XEHR,6.7582,Striped Men Polo Neck White T-Shirt,NaN
3,4,TSHFZKJHBGGUBASG,6.7582,Solid Men Polo Neck Grey T-Shirt,NaN
4,5,TSHFZKJHQ4GWPCDY,6.7582,Solid Men Polo Neck Grey T-Shirt,NaN
5,6,TSHFUGNWKWGPZ32S,6.7245,Solid Men Polo Neck Black T-Shirt,1.0
6,7,TSHFUGZ9ZG5XN2FY,6.7245,Solid Men Polo Neck Blue T-Shirt,1.0
7,8,TSHFZHPBVYPHHC2J,6.7158,Solid Men Polo Neck Pink T-Shirt,NaN
8,9,TSHFZHPBXMRMMUY4,6.7158,Solid Men Polo Neck Green T-Shirt,NaN
9,10,TSHFZKJHASFMNT2Y,6.7158,Solid Men Polo Neck Grey T-Shirt,NaN


Query 2: women cotton kurta | TF-IDF + cosine


,rank,pid,score,title,label
0,1,KTAFAKGUDJEB5DR7,0.5590,Women Solid Cotton Blend Straight Kurta (Black),NaN
1,2,KTAFH7NBG8PEX5GP,0.5186,"Women Solid Cotton Blend Straight Kurta (Maroon, Yellow)",NaN
2,3,KTAFMRFBSZ5QYCUW,0.5099,Women Solid Pure Cotton Straight Kurta (Black),NaN
3,4,KTAFAJFFHXVJHUMU,0.5094,Women Solid Cotton Blend Straight Kurta (Green),1.0
4,5,KTAFZM3FNPJS7QUK,0.5039,Women Solid Cotton Blend Straight Kurta (Dark Blue),NaN
5,6,KTAFZM3FPYWFMP7W,0.4987,Women Solid Cotton Blend Straight Kurta (Yellow),NaN
6,7,KTAFAJFFZYZTVNRA,0.4984,Women Solid Cotton Blend Straight Kurta (White),1.0
7,8,FABFWN82HVT5KAH2,0.4920,Cotton Printed Kurta Fabric (Unstitched),NaN
8,9,ETHFPDUMHA2BTSZH,0.4916,Women Kurta and Pyjama Set Pure Cotton,NaN
9,10,ETHFPDUMHFZMYHCX,0.4916,Women Kurta and Pyjama Set Pure Cotton,NaN


... (10 outputs left out)

Query 5: women blue denim jacket | TF-IDF + cosine


,rank,pid,score,title,label
0,1,JCKFZ33GHPEXHFUZ,0.9084,Full Sleeve Solid Women Jacket,1
1,2,JCKFWC5YGQENYT6B,0.7812,Full Sleeve Solid Women Denim Jacket,1
2,3,JCKFYHVRWMSWQJ2J,0.7587,Full Sleeve Solid Women Jacket,1
3,4,JCKFSHEZTYYPEMTJ,0.7408,Full Sleeve Solid Women Denim Jacket,1
4,5,JCKFVDPXFVQBGME7,0.7393,Full Sleeve Solid Women Denim Jacket,1
5,6,JCKFXNVMGBYGXKHN,0.7386,Full Sleeve Solid Women Denim Jacket,1
6,7,JCKFVEUDNF3J7CWV,0.7321,Full Sleeve Solid Women Jacket,1
7,8,JCKFVEUDBT4KKSXR,0.7196,Full Sleeve Solid Women Jacket,1
8,9,JCKFSJZVVFNUYTZW,0.6535,Full Sleeve Self Design Women Jacket,1
9,10,JCKF7KY3NQUGDHE2,0.6322,Full Sleeve Solid Women Denim Casual Jacket,1


Query 5: women blue denim jacket | BM25


,rank,pid,score,title,label
0,1,JCKFZ33GHPEXHFUZ,13.1808,Full Sleeve Solid Women Jacket,1
1,2,JCKFWC5YGQENYT6B,13.0658,Full Sleeve Solid Women Denim Jacket,1
2,3,JCKFSHEZTYYPEMTJ,12.8926,Full Sleeve Solid Women Denim Jacket,1
3,4,JCKFVDPXFVQBGME7,12.5812,Full Sleeve Solid Women Denim Jacket,1
4,5,JCKFXNVMGBYGXKHN,12.5812,Full Sleeve Solid Women Denim Jacket,1
5,6,JCKFVEUDNF3J7CWV,12.3904,Full Sleeve Solid Women Jacket,1
6,7,JCKFVEUDBT4KKSXR,12.2949,Full Sleeve Solid Women Jacket,1
7,8,JCKFV7V7ZQ3PUSFF,12.2859,Full Sleeve Washed Women Denim Riding Jacket,1
8,9,JCKF7KY3NQUGDHE2,12.2065,Full Sleeve Solid Women Denim Casual Jacket,1
9,10,JCKFWQE5VX2YHF9J,12.2010,Full Sleeve Animal Print Women Jacket,1


### How different are the two rankings?

* **overlap**: products that are in both top 20
* **median length**: median number of tokens of the top 20 products
* **labelled 1 / 0 / not labelled**: our Part 2 labels only cover the top 20 of the Part 2 ranking,
  so products that are new in the top 20 have no label

In [6]:
rows = []
for qid, q in TEST_QUERIES.items():
    n_matches = len(conjunctive_search(preprocess_text(q), index))
    tfidf_res, bm25_res = results[(qid, "tfidf")], results[(qid, "bm25")]
    row = {"query": q, "AND matches": n_matches,
           "overlap": len(set(tfidf_res.pid) & set(bm25_res.pid))}
    for name, res in [("tfidf", tfidf_res), ("bm25", bm25_res)]:
        row[f"{name} median length"] = int(np.median([data["doc_lengths"][p] for p in res.pid]))
        row[f"{name} labelled 1/0/none"] = f"{int((res.label == 1).sum())}/{int((res.label == 0).sum())}/{int(res.label.isna().sum())}"
    rows.append(row)
comparison = pd.DataFrame(rows, index=pd.Index(TEST_QUERIES.keys(), name="query_id"))
comparison

,query,AND matches,overlap,tfidf median length,tfidf labelled 1/0/none,bm25 median length,bm25 labelled 1/0/none
query_id,,,,,,,
1,men cotton polo tshirt,1213,4,21,3/0/17,26,6/0/14
2,women cotton kurta,358,3,23,2/0/18,24,3/0/17
3,men hooded sweatshirt,323,0,19,0/0/20,29,18/0/2
4,men running shoes,37,18,27,11/6/3,31,11/8/1
5,women blue denim jacket,27,20,27,13/7/0,27,13/7/0


* With few AND matches (queries 4 and 5) both methods return almost the same products.
* With many near-identical matches (queries 1-3) the top 20 is very different, and most of those
  products were never labelled in Part 2.
* Both methods move the pack of socks of "men running shoes" down, from rank 4 in Part 2 to rank 13
  (TF-IDF) and 11 (BM25).
* TF-IDF + cosine puts an unstitched kurta **fabric** at rank 8 of "women cotton kurta": a short text
  with all the query terms wins even if it is not the product the user wants.

### Why do they disagree? Example: "men hooded sweatshirt"

The two methods share **no** product in their top 20. Top 3 of each, with the length of the product
and how many times it contains each query term:

In [7]:
q = "men hooded sweatshirt"
terms = preprocess_text(q)
for method in ["tfidf", "bm25"]:
    for pid, score in search_ranked(q, index, idf, data, method=method, k=3):
        counts = {t: data["counts"][pid][t] for t in terms}
        print(f"{method:5} {score:.4f}  {title[pid]:38} length {data['doc_lengths'][pid]:3}  {counts}")

tfidf 0.8191  Full Sleeve Solid Men Sweatshirt       length  20  {'men': 1, 'hood': 2, 'sweatshirt': 2}
tfidf 0.8175  Full Sleeve Solid Men Sweatshirt       length  20  {'men': 1, 'hood': 2, 'sweatshirt': 2}
tfidf 0.8146  Full Sleeve Solid Men Sweatshirt       length  20  {'men': 1, 'hood': 2, 'sweatshirt': 2}
bm25  12.2004  Full Sleeve Printed Men Sweatshirt     length  32  {'men': 3, 'hood': 4, 'sweatshirt': 4}
bm25  12.2004  Full Sleeve Printed Men Sweatshirt     length  32  {'men': 3, 'hood': 4, 'sweatshirt': 4}
bm25  12.0873  Full Sleeve Printed Men Sweatshirt     length  34  {'men': 3, 'hood': 4, 'sweatshirt': 4}


* **TF-IDF + cosine** divides by the length of the whole product vector, so it prefers the **shortest**
  products, where the query terms are a big part of the text (20 tokens, each term once or twice).
* **BM25** rewards products that **repeat** the query terms more (here 3-4 times), and only penalises
  length a little: these products have 32 tokens, close to the average of 39.

### Summary: TF-IDF + cosine vs BM25

| | TF-IDF + cosine | BM25 |
|---|---|---|
| repeated terms | grow with log, without limit | saturate ($k_1$) |
| length | divides by the length of the whole tf-idf vector, so short products win | soft penalty relative to the average length ($b$) |
| parameters | none | $k_1$ and $b$, must be tuned |
| when the AND leaves few products (Q4, Q5) | almost the same top 20 as BM25 | almost the same top 20 as TF-IDF |
| when it leaves many near-identical products (Q1–Q3) | very different top 20 | very different top 20 |

## 2. Our score

Our score starts from BM25 and multiplies it by four factors built from the fields of Part 1.
The code is in `myapp/search/custom_score.py`.

$$score = BM25 \cdot f_{title} \cdot f_{rating} \cdot f_{stock} \cdot f_{discount}$$

| factor | formula | range | why |
|---|---|---|---|
| title | $1 + \frac{\text{query terms in the title}}{\text{query terms}}$ | 1 – 2 | Part 1: the title is the most discriminative field; Part 2: socks and fabrics matched only outside the title |
| rating | $1 + 0.25 \cdot \frac{r - \bar r}{\bar r}$, 1 if there is no rating | 0.82 – 1.09 | better rated products first, with a small weight because there is no number of reviews |
| stock | 0.5 if out of stock, else 1 | 0.5 or 1 | Part 1 decision: demote unavailable products, do not remove them |
| discount | $1 + 0.2 \cdot \frac{discount}{100}$ | 1 – 1.17 | small bonus; list prices of cheap products are often inflated (Part 1) |

BM25 still decides which products are about the query; the factors mostly reorder products whose
BM25 scores are close, which in this corpus are often the same garment in other colours.

In [8]:
from myapp.search.custom_score import create_custom_data, search_custom, custom_factors

custom_data = create_custom_data(corpus)
print(f"mean rating: {custom_data['mean_rating']:.2f}")

mean rating: 3.63


In [9]:
def top20_custom(query):
    #same table as top20() but with our score
    labels = test_labels[test_labels["query"] == query].set_index("pid")["label"]
    res = pd.DataFrame(search_custom(query, index, idf, data, custom_data, k=20), columns=["pid", "score"])
    res.insert(0, "rank", np.arange(1, len(res) + 1))
    res["title"] = res.pid.map(title)
    res["label"] = res.pid.map(labels)
    return res.round({"score": 4})

for qid, q in TEST_QUERIES.items():
    results[(qid, "custom")] = top20_custom(q)
    print(f"Query {qid}: {q} | our score")
    display(results[(qid, "custom")])

Query 1: men cotton polo tshirt | our score


,rank,pid,score,title,label
0,1,TSHFZQBG3J5GREG2,13.5004,Printed Men Polo Neck Black T-Shirt,NaN
1,2,TSHFZQBGBQGXVRHJ,13.4882,Printed Men Polo Neck Grey T-Shirt,NaN
2,3,TSHFUGKTXHZTUGCH,13.4005,Striped Men Polo Neck Blue T-Shirt,1.0
3,4,TSHFUGNWKWGPZ32S,13.3720,Solid Men Polo Neck Black T-Shirt,1.0
4,5,TSHFUGZ9ZG5XN2FY,13.3720,Solid Men Polo Neck Blue T-Shirt,1.0
5,6,TSHFUGKSKZUGEYYQ,13.3173,Striped Men Polo Neck Red T-Shirt,1.0
6,7,TSHFUNN36MB6BAPG,13.3011,Solid Men Polo Neck Yellow T-Shirt,NaN
7,8,TSHFZKJHBGGUBASG,13.2949,Solid Men Polo Neck Grey T-Shirt,NaN
8,9,TSHFZKJHQ4GWPCDY,13.2949,Solid Men Polo Neck Grey T-Shirt,NaN
9,10,TSHFUGP7GVGXJWTC,13.2884,Striped Men Polo Neck Green T-Shirt,1.0


Query 2: women cotton kurta | our score


,rank,pid,score,title,label
0,1,KTAF85TUHDNZCU3P,18.3593,Women Solid Cotton Blend Straight Kurta (Purple),NaN
1,2,KTAFAJFFZYZTVNRA,18.3170,Women Solid Cotton Blend Straight Kurta (White),1.0
2,3,KTAFZPA5GRMYWAJ9,18.2079,Women Striped Cotton Blend Straight Kurta (Yellow),NaN
3,4,KTAF8FJCZVQ7QHH9,17.9115,Women Solid Cotton Blend Pathani Kurta (Black),NaN
4,5,KTAFH7NBG8PEX5GP,17.8420,"Women Solid Cotton Blend Straight Kurta (Maroon, Yellow)",NaN
5,6,KTAFZFX7HTMGQCG6,17.6794,Women Solid Cotton Blend Straight Kurta (Black),NaN
6,7,KTAF8FJVVYE2EDWN,17.5396,Women Solid Cotton Blend Straight Kurta (Yellow),1.0
7,8,KTAFZFX7MGG3ZTYP,17.5380,Women Solid Cotton Blend Straight Kurta (Red),NaN
8,9,KTAFWG3KHPEVYFRG,17.5175,Women Solid Pure Cotton Straight Kurta (Maroon),NaN
9,10,KTAFWG3KMZYPJ7NJ,17.5175,Women Solid Pure Cotton Straight Kurta (Green),NaN


Query 3: men hooded sweatshirt | our score


,rank,pid,score,title,label
0,1,SWSFJE3V7A8FHGNG,23.6857,Sleeveless Printed Men Sweatshirt,1.0
1,2,SWSFYFFFMWQMAPJJ,22.2212,Full Sleeve Printed Men Sweatshirt,1.0
2,3,SWSFYFFYJCSGNGD8,22.2212,Full Sleeve Printed Men Sweatshirt,1.0
3,4,SWSFYFFFZBZKUQUS,22.1325,Full Sleeve Printed Men Sweatshirt,1.0
4,5,SWSFXMFPVYZ2YGGQ,22.0458,Full Sleeve Embroidered Men Sweatshirt,1.0
5,6,SWSFXMFPYYTZUZ3Z,21.9515,Full Sleeve Printed Men Sweatshirt,1.0
6,7,SWSFZUEBYU5DM9CH,21.9301,Full Sleeve Printed Men Sweatshirt,NaN
7,8,SWSFYE8BSGHCMCCG,21.8707,Full Sleeve Printed Men Sweatshirt,NaN
8,9,SWSFYE8DVWRPTQWH,21.8707,Full Sleeve Printed Men Sweatshirt,NaN
9,10,SWSFPHFFKEGXGVHW,21.8163,Full Sleeve Solid Men Sweatshirt,NaN


Query 4: men running shoes | our score


,rank,pid,score,title,label
0,1,SHOFTZFQQYKC8HY6,30.1718,Running Shoes For Men (Yellow),1.0
1,2,SHOFTZFPNCU4VWHK,30.0116,Running Shoes For Men (Black),1.0
2,3,SHOESS9ZEZ6TEQUJ,28.4831,Running Shoes For Men (Multicolor),1.0
3,4,SHOESS9ZGMH7CURM,27.3575,Running Shoes For Men (Multicolor),1.0
4,5,SHOFN3EJVGC9VXYT,26.9570,RS-7200 Running Shoes For Men (Navy),1.0
5,6,SHOFN3EJHXHGSGUC,26.7142,RS-7250 Running Shoes For Men (Black),1.0
6,7,SHOFN3EFNYGBKZT7,26.4764,"RS-7050 Running Shoes For Men (Blue, Navy)",1.0
7,8,SHOEKNRJ34UCMAPA,25.7576,"SPRINT Running Shoes For Men (Black, Grey)",1.0
8,9,SHOEKNRJGEYJCDQK,25.7576,"SPRINT Running Shoes For Men (Blue, Grey)",1.0
9,10,SHOEWQKVSGFGXPQF,25.0511,Running Shoes For Men (Blue),1.0


Query 5: women blue denim jacket | our score


,rank,pid,score,title,label
0,1,JCKFWC5YGQENYT6B,25.9376,Full Sleeve Solid Women Denim Jacket,1.0
1,2,JCKFVDPXFVQBGME7,24.4722,Full Sleeve Solid Women Denim Jacket,1.0
2,3,JCKFV7V7ZQ3PUSFF,24.3347,Full Sleeve Washed Women Denim Riding Jacket,1.0
3,4,JCKFSHEZTYYPEMTJ,23.9558,Full Sleeve Solid Women Denim Jacket,1.0
4,5,JCKFVEUDNF3J7CWV,21.3664,Full Sleeve Solid Women Jacket,1.0
5,6,JCKFVEUDBT4KKSXR,21.2017,Full Sleeve Solid Women Jacket,1.0
6,7,JCKFZ33GHPEXHFUZ,20.6806,Full Sleeve Solid Women Jacket,1.0
7,8,JCKFWQE5VX2YHF9J,20.4283,Full Sleeve Animal Print Women Jacket,1.0
8,9,JCKFMHS8ANSPXK72,19.8554,Full Sleeve Printed Women Jacket,1.0
9,10,JCKFSJZVVFNUYTZW,17.5832,Full Sleeve Self Design Women Jacket,1.0


### Our score vs BM25

* **out of stock / mean rating**: in the top 20
* **title factor**: mean of $f_{title}$ in the top 20 (2 = all query terms in the title)

In [10]:
products = custom_data["products"]
rows = []
for qid, q in TEST_QUERIES.items():
    terms = preprocess_text(q)
    bm25_res, custom_res = results[(qid, "bm25")], results[(qid, "custom")]
    row = {"query": q, "overlap": len(set(bm25_res.pid) & set(custom_res.pid))}
    for name, res in [("bm25", bm25_res), ("custom", custom_res)]:
        ratings = [products[p]["rating"] for p in res.pid if products[p]["rating"] is not None]
        row[f"{name} labelled 1/0/none"] = f"{int((res.label == 1).sum())}/{int((res.label == 0).sum())}/{int(res.label.isna().sum())}"
        row[f"{name} out of stock"] = sum(products[p]["out_of_stock"] for p in res.pid)
        row[f"{name} mean rating"] = round(np.mean(ratings), 2)
        row[f"{name} title factor"] = round(np.mean([custom_factors(p, terms, custom_data)["title"] for p in res.pid]), 2)
    rows.append(row)
custom_comparison = pd.DataFrame(rows, index=pd.Index(TEST_QUERIES.keys(), name="query_id"))
custom_comparison

,query,overlap,bm25 labelled 1/0/none,bm25 out of stock,bm25 mean rating,bm25 title factor,custom labelled 1/0/none,custom out of stock,custom mean rating,custom title factor
query_id,,,,,,,,,,
1,men cotton polo tshirt,13,6/0/14,2,3.69,1.75,7/0/13,0,4.16,1.75
2,women cotton kurta,7,3/0/17,0,3.51,2.00,7/0/13,0,4.31,2.00
3,men hooded sweatshirt,10,18/0/2,0,3.37,1.67,10/0/10,0,3.91,1.67
4,men running shoes,11,11/8/1,2,3.88,1.70,11/0/9,0,3.94,2.00
5,women blue denim jacket,19,13/7/0,3,3.66,1.55,13/6/1,2,3.69,1.56


### The two problems of Part 2

Position of the pack of socks in "men running shoes" and of the unstitched kurta fabric in
"women cotton kurta", over the full list of AND matches:

In [11]:
def full_rank(query, method, words):
    #positions (1 = first) of the products whose title contains `words`
    if method == "custom":
        ranked = search_custom(query, index, idf, data, custom_data, k=None)
    else:
        ranked = search_ranked(query, index, idf, data, method=method, k=None)
    return [i + 1 for i, (pid, _) in enumerate(ranked) if words in title[pid]]

pd.DataFrame([
    {"product": "socks (Peds/Footie)", "query": "men running shoes",
     **{m: full_rank("men running shoes", m, "Peds") for m in ["tfidf", "bm25", "custom"]}},
    {"product": "kurta fabric", "query": "women cotton kurta",
     **{m: full_rank("women cotton kurta", m, "Fabric") for m in ["tfidf", "bm25", "custom"]}},
])

,product,query,tfidf,bm25,custom
0,socks (Peds/Footie),men running shoes,[13],[11],[22]
1,kurta fabric,women cotton kurta,[8],[61],[329]


Both products have the query terms mostly outside the title, so the title factor pushes them down:
the socks leave the top 20 and the fabric falls to the end of the list.

### The price of the stock factor: "women blue denim jacket"

BM25 top 20 with our factors, and the new position of each product:

In [12]:
q = "women blue denim jacket"
terms = preprocess_text(q)
labels_q = test_labels[test_labels["query"] == q].set_index("pid")["label"]
custom_order = [pid for pid, _ in search_custom(q, index, idf, data, custom_data, k=None)]
rows = []
for rank, (pid, score) in enumerate(search_ranked(q, index, idf, data, method="bm25", k=20), start=1):
    factors = custom_factors(pid, terms, custom_data)
    rows.append({"bm25 rank": rank, "our rank": custom_order.index(pid) + 1, "label": labels_q.get(pid),
                 "title": title[pid], **{f"f_{k}": round(v, 2) for k, v in factors.items()}})
pd.DataFrame(rows)

,bm25 rank,our rank,label,title,f_title,f_rating,f_stock,f_discount
0,1,7,1,Full Sleeve Solid Women Jacket,1.50,1.00,1.0,1.05
1,2,1,1,Full Sleeve Solid Women Denim Jacket,1.75,1.03,1.0,1.11
2,3,4,1,Full Sleeve Solid Women Denim Jacket,1.75,0.94,1.0,1.13
3,4,2,1,Full Sleeve Solid Women Denim Jacket,1.75,1.00,1.0,1.11
4,5,16,1,Full Sleeve Solid Women Denim Jacket,1.75,1.05,0.5,1.11
5,6,5,1,Full Sleeve Solid Women Jacket,1.50,1.04,1.0,1.11
6,7,6,1,Full Sleeve Solid Women Jacket,1.50,1.04,1.0,1.11
7,8,3,1,Full Sleeve Washed Women Denim Riding Jacket,1.75,1.00,1.0,1.13
8,9,17,1,Full Sleeve Solid Women Denim Casual Jacket,1.75,0.82,0.5,1.14
9,10,8,1,Full Sleeve Animal Print Women Jacket,1.50,0.98,1.0,1.14


Two relevant denim jackets (BM25 ranks 5 and 9) are out of stock, so they fall to ranks 16 and 17
and four blue T-shirts move above them. This is what the stock factor is meant to do, but our
Part 2 labels only judged the product, not whether it can be bought, so with those labels it looks
like an error.

### Summary: pros and cons of our score

| pros | cons |
|---|---|
| uses the title, the field that best separates relevant products (fixes the socks and the fabric) | the weights (1, 0.25, 0.5, 0.2) are chosen by hand, not learned from labels |
| demotes products that cannot be bought | a relevant product that is out of stock goes down ("women blue denim jacket") |
| breaks the many near-ties between identical garments with a quality signal | the rating has no number of reviews, so it is noisy |
| every factor is easy to explain and to switch off | the title factor counts all query terms equally: "men" in the title counts as much as "shoe" |
| cheap: all the values are precomputed once | the discount can be inflated on cheap products |

## 3. word2vec + cosine similarity

We train word2vec ourselves with numpy (skip-gram with negative sampling, 100 dimensions, window 5,
5 epochs) on our own products, using the processed tokens of Part 1. The code is in
`myapp/search/word2vec_score.py`.

**Training.** For every pair of words that appear close to each other in a product (a word and its
context), the model moves their vectors closer; for 5 random words (negative examples) it moves them
apart. After a few passes, words used in similar contexts have similar vectors. Very frequent words
are randomly skipped (subsampling), because they appear everywhere and say little.

**Ranking.**
* vector of a text: average of the vectors of its words, $\vec d = \frac{1}{|d|}\sum_{t \in d} \vec w_t$
* score: $\cos(\vec q, \vec d) = \dfrac{\vec q \cdot \vec d}{\lVert \vec q \rVert \cdot \lVert \vec d \rVert}$

We keep the same AND filter, so only the order changes. Every candidate contains all the query terms,
so the score mostly measures how close the **rest** of the product text is to the meaning of the query.

In [ ]:
from myapp.search.word2vec_score import train_word2vec, save_word_vectors, load_word_vectors, create_word2vec_data, search_word2vec

#training takes a few minutes, so we save the vectors and reuse them in the next runs
VECTORS_PATH = DATA / "word2vec_vectors.npz"
if VECTORS_PATH.exists():
    wv = load_word_vectors(VECTORS_PATH)
else:
    wv = train_word2vec(corpus)
    save_word_vectors(wv, VECTORS_PATH)

w2v_data = create_word2vec_data(corpus, wv)
print(f"vocabulary: {len(wv.words):,} words with a vector (of {len(idf):,} terms in the index)")

word2vec: epoch 1/5 done
word2vec: epoch 2/5 done
word2vec: epoch 3/5 done
word2vec: epoch 4/5 done
word2vec: epoch 5/5 done
vocabulary: 4,440 words with a vector (of 5,710 terms in the index)


### Top 20 of our five queries

In [ ]:
def top20_w2v(query):
    #same table as top20() but with word2vec + cosine
    labels = test_labels[test_labels["query"] == query].set_index("pid")["label"]
    res = pd.DataFrame(search_word2vec(query, index, w2v_data, k=20), columns=["pid", "score"])
    res.insert(0, "rank", np.arange(1, len(res) + 1))
    res["title"] = res.pid.map(title)
    res["label"] = res.pid.map(labels)
    return res.round({"score": 4})

for qid, q in TEST_QUERIES.items():
    results[(qid, "w2v")] = top20_w2v(q)
    print(f"Query {qid}: {q} | word2vec + cosine")
    display(results[(qid, "w2v")])

Query 1: men cotton polo tshirt | word2vec + cosine


,rank,pid,score,title,label
0,1,TSHFZKJHBGGUBASG,0.9239,Solid Men Polo Neck Grey T-Shirt,NaN
1,2,TSHFZKJHQ4GWPCDY,0.9239,Solid Men Polo Neck Grey T-Shirt,NaN
2,3,TSHFUNN3HZKFZZDJ,0.9232,Printed Men Polo Neck Grey T-Shirt,NaN
3,4,TSHFUNN36MB6BAPG,0.9204,Solid Men Polo Neck Yellow T-Shirt,NaN
4,5,TSHFZKJHASFMNT2Y,0.9190,Solid Men Polo Neck Grey T-Shirt,NaN
5,6,TSHFZHPBVYPHHC2J,0.9184,Solid Men Polo Neck Pink T-Shirt,NaN
6,7,TSHFZKJHPDKJZZ8X,0.9178,Solid Men Polo Neck Grey T-Shirt,NaN
7,8,TSHFZHPBXMRMMUY4,0.9164,Solid Men Polo Neck Green T-Shirt,NaN
8,9,TSHFUGNWKWGPZ32S,0.9158,Solid Men Polo Neck Black T-Shirt,1.0
9,10,TSHFUNN2YHM6XEHR,0.9145,Striped Men Polo Neck White T-Shirt,NaN


Query 2: women cotton kurta | word2vec + cosine


,rank,pid,score,title,label
0,1,KTAFHRQG6SHX9Q4H,0.8815,Women Solid Cotton Blend Straight Kurta (Purple),NaN
1,2,KTAFHRQGCB3BFPBU,0.8815,Women Solid Cotton Blend Straight Kurta (Purple),NaN
2,3,KTAFHRQGGMVF29YQ,0.8815,Women Solid Cotton Blend Straight Kurta (Purple),NaN
3,4,KTAFHRQGKGEVE4KQ,0.8815,Women Solid Cotton Blend Straight Kurta (Purple),NaN
4,5,KTAFHRQGVTRHGYB7,0.8815,Women Solid Cotton Blend Straight Kurta (Purple),NaN
5,6,KTAFHRQGZUAMYB62,0.8815,Women Solid Cotton Blend Straight Kurta (Purple),NaN
6,7,KTAFAJFFZYZTVNRA,0.8814,Women Solid Cotton Blend Straight Kurta (White),1.0
7,8,KTAF85TUHDNZCU3P,0.8809,Women Solid Cotton Blend Straight Kurta (Purple),NaN
8,9,KTAFH7NBG8PEX5GP,0.8806,"Women Solid Cotton Blend Straight Kurta (Maroon, Yellow)",NaN
9,10,KTAFAJFFHXVJHUMU,0.8766,Women Solid Cotton Blend Straight Kurta (Green),1.0


Query 3: men hooded sweatshirt | word2vec + cosine


,rank,pid,score,title,label
0,1,SWSFYE8BSGHCMCCG,0.8911,Full Sleeve Printed Men Sweatshirt,NaN
1,2,SWSFYE8DVWRPTQWH,0.8877,Full Sleeve Printed Men Sweatshirt,NaN
2,3,SWSFXMFPRTY4GKGF,0.8842,Full Sleeve Printed Men Sweatshirt,1.0
3,4,SWSFUNF6G7RZZ9FD,0.8813,Full Sleeve Printed Men Sweatshirt,NaN
4,5,SWSFUNF6TYFXREER,0.8813,Full Sleeve Printed Men Sweatshirt,NaN
5,6,SWSFUNF6VFJBYZZC,0.8813,Full Sleeve Printed Men Sweatshirt,NaN
6,7,SWSFUNF6XHCGGVCU,0.8813,Full Sleeve Printed Men Sweatshirt,NaN
7,8,SWSFUNF6ZCFUZKHR,0.8813,Full Sleeve Printed Men Sweatshirt,NaN
8,9,SWSFVZPXB9K4K3G5,0.8789,Full Sleeve Printed Men Sweatshirt,NaN
9,10,SWSFVZPXDZXCGAZW,0.8789,Full Sleeve Printed Men Sweatshirt,NaN


Query 4: men running shoes | word2vec + cosine


,rank,pid,score,title,label
0,1,SHOESS9ZEZ6TEQUJ,0.8967,Running Shoes For Men (Multicolor),1.0
1,2,SHOESS9ZGMH7CURM,0.8967,Running Shoes For Men (Multicolor),1.0
2,3,SHOEKNRJGEYJCDQK,0.8683,"SPRINT Running Shoes For Men (Blue, Grey)",1.0
3,4,SHOFN3EJVGC9VXYT,0.8683,RS-7200 Running Shoes For Men (Navy),1.0
4,5,SHOFN3EJHXHGSGUC,0.8648,RS-7250 Running Shoes For Men (Black),1.0
5,6,SHOFN3EFNYGBKZT7,0.8627,"RS-7050 Running Shoes For Men (Blue, Navy)",1.0
6,7,SHOEKNRJ34UCMAPA,0.8611,"SPRINT Running Shoes For Men (Black, Grey)",1.0
7,8,SHOFTZFPNCU4VWHK,0.8601,Running Shoes For Men (Black),1.0
8,9,SHOFTZFQQYKC8HY6,0.8592,Running Shoes For Men (Yellow),1.0
9,10,SHOEWQKVSGFGXPQF,0.8582,Running Shoes For Men (Blue),1.0


Query 5: women blue denim jacket | word2vec + cosine


,rank,pid,score,title,label
0,1,JCKFZ33GHPEXHFUZ,0.9005,Full Sleeve Solid Women Jacket,1.0
1,2,JCKFWC5YGQENYT6B,0.8984,Full Sleeve Solid Women Denim Jacket,1.0
2,3,JCKFSHEZTYYPEMTJ,0.8969,Full Sleeve Solid Women Denim Jacket,1.0
3,4,JCKFVDPXFVQBGME7,0.8836,Full Sleeve Solid Women Denim Jacket,1.0
4,5,JCKFXNVMGBYGXKHN,0.8823,Full Sleeve Solid Women Denim Jacket,1.0
5,6,JCKF7KY3NQUGDHE2,0.8781,Full Sleeve Solid Women Denim Casual Jacket,1.0
6,7,JCKFWQE5VX2YHF9J,0.8702,Full Sleeve Animal Print Women Jacket,1.0
7,8,JCKFYHVRWMSWQJ2J,0.8676,Full Sleeve Solid Women Jacket,1.0
8,9,JCKFV7V7ZQ3PUSFF,0.8667,Full Sleeve Washed Women Denim Riding Jacket,1.0
9,10,JCKFVEUDBT4KKSXR,0.8658,Full Sleeve Solid Women Jacket,1.0


### word2vec vs TF-IDF and BM25

* **overlap**: products that are in both top 20
* **median length**: median number of tokens of the top 20 products
* **labelled 1 / 0 / not labelled**: our Part 2 labels

In [ ]:
rows = []
for qid, q in TEST_QUERIES.items():
    w2v_res = results[(qid, "w2v")]
    rows.append({
        "query": q,
        "overlap tfidf": len(set(w2v_res.pid) & set(results[(qid, "tfidf")].pid)),
        "overlap bm25": len(set(w2v_res.pid) & set(results[(qid, "bm25")].pid)),
        "w2v median length": int(np.median([data["doc_lengths"][p] for p in w2v_res.pid])),
        "w2v labelled 1/0/none": f"{int((w2v_res.label == 1).sum())}/{int((w2v_res.label == 0).sum())}/{int(w2v_res.label.isna().sum())}",
    })
w2v_comparison = pd.DataFrame(rows, index=pd.Index(TEST_QUERIES.keys(), name="query_id"))
w2v_comparison

,query,overlap tfidf,overlap bm25,overlap w2v idf-weighted,w2v median length,w2v labelled 1/0/none
query_id,,,,,,
1,men cotton polo tshirt,6,14,13,25,4/0/16
2,women cotton kurta,9,6,12,45,4/0/16
3,men hooded sweatshirt,0,3,14,21,3/0/17
4,men running shoes,17,18,19,31,11/8/1
5,women blue denim jacket,19,19,20,27,13/6/1


* With few AND matches (queries 4 and 5) word2vec returns almost the same top 20 as TF-IDF and BM25
  (17–19 products in common).
* With many near-identical matches (queries 1–3) the top 20 changes again: "men hooded sweatshirt"
  shares no product with TF-IDF and only 3 with BM25. Most of these products were not labelled in Part 2.
* Unlike TF-IDF + cosine, it does not prefer short products, because an average does not depend on the
  length: the median length for "women cotton kurta" is 45 tokens, against 23 with TF-IDF.
* Ordering follows the type of product: in "men running shoes" the 11 running shoes come first and then
  sneakers, loafers and party shoes; in "women blue denim jacket" the first 12 are all jackets and the
  blue T-shirts come after.
* Products with the same text get the same vector and the same score: the first six results of
  "women cotton kurta" are the same purple kurta (0.8815), ordered only by pid.

### The two problems of Part 2

Position of the pack of socks in "men running shoes" and of the unstitched kurta fabric in
"women cotton kurta", over the full list of AND matches:

In [ ]:
def full_rank_w2v(query, words):
    #positions (1 = first) of the products whose title contains `words`
    ranked = search_word2vec(query, index, w2v_data, k=None)
    return [i + 1 for i, (pid, _) in enumerate(ranked) if words in title[pid]]

pd.DataFrame([
    {"product": "socks (Peds/Footie)", "query": "men running shoes", "w2v": full_rank_w2v("men running shoes", "Peds")},
    {"product": "kurta fabric", "query": "women cotton kurta", "w2v": full_rank_w2v("women cotton kurta", "Fabric")},
])

,product,query,w2v,w2v idf
0,socks (Peds/Footie),men running shoes,[34],[34]
1,kurta fabric,women cotton kurta,[330],[273]


word2vec puts the socks at rank 34 (TF-IDF 13, BM25 11, our score 22) and the fabric at rank 330
(our score 329) **without using the title**: the average vector of a pack of socks or of a fabric is
about socks or fabrics, so it is far from the meaning of the query, even if it contains all its words.

### Summary: pros and cons of word2vec + cosine

| pros | cons |
|---|---|
| captures meaning: words used in similar contexts get close vectors | averaging loses word order, and generic words pull every product toward the same "average fashion" vector |
| ranks by what the product is: pushes the socks and the fabric down without using the title | products with the same text get exactly the same score, so many ties |
| does not prefer short products: the average does not depend on the length | needs training (size, window, epochs...) and the vectors depend on the random seed |
| trained on our own corpus, so it learns the fashion vocabulary | 1,270 rare terms have no vector: 28k products is a small corpus for word2vec |
| | with the AND filter it can only reorder products, so its main advantage (finding products without the exact words) is not used |